In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim

In [3]:
spark = get_sparkSession("Load silver.product_price")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "product_price"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_product_price.ipynb"
_bronze_table = "bronze.product_price"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.product_price


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze.raw_customer and de-duplicating the data

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

df_dedup = df.withColumn("_rn", expr("row_number() over (partition by product_id, store, channel, effective_start_date sort by created_on desc)")) \
             .where("_rn = 1") \
             .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Bronze Table Data count : 3
SPARK-APP: Table count after de-dupe : 3


In [10]:
df.show(10)
df.printSchema()


+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------+--------------------+--------------------+--------------------+
|product_id| store|channel|list_price|cost_price|currency|effective_start_date|is_current|is_deleted|    source_file|          created_on|          created_by|         modified_on|         modified_by|
+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------+--------------------+--------------------+--------------------+
|     P1001|AMZ_US|    MKT|       500|       350|     USD|          2025-01-01|      true|     false|dim_product.csv|2026-01-29 00:21:...|raw_product_price...|2026-01-29 00:21:...|raw_product_price...|
|     P1002|  EBAY|    MKT|      1200|       900|     USD|          2025-02-15|      true|     false|dim_product.csv|2026-01-29 00:21:...|raw_product_price...|2026-01-29 00:21:...|raw_product_

In [11]:
#Formating the data

df_ld = df_dedup.withColumn("effective_start_date", to_date("effective_start_date", 'yyyy-MM-dd')) \
                .withColumn("list_price", expr("CAST(list_price as decimal(10,2))")) \
                .withColumn("cost_price", expr("CAST(cost_price as decimal(10,2))")) \
                .withColumn("is_current", when(lower("is_current") == "true", True).otherwise(False)) \
                .withColumn("is_deleted", when(lower("is_deleted") == "true", True).otherwise(False))
                

In [12]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("product_id", upper(trim("product_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel"))) \
             .withColumn("currency", upper(trim("currency"))) 

In [13]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [14]:
df_ld.show(10)
df_ld.printSchema()

+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------+--------------------+--------------------+--------------------+
|product_id| store|channel|list_price|cost_price|currency|effective_start_date|is_current|is_deleted|    source_file|          created_on|          created_by|         modified_on|         modified_by|
+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------+--------------------+--------------------+--------------------+
|     P1001|AMZ_US|    MKT|    500.00|    350.00|     USD|          2025-01-01|      true|     false|dim_product.csv|2026-01-29 01:12:...|silver_product_pr...|2026-01-29 01:12:...|silver_product_pr...|
|     P1002|  EBAY|    MKT|   1200.00|    900.00|     USD|          2025-02-15|      true|     false|dim_product.csv|2026-01-29 01:12:...|silver_product_pr...|2026-01-29 01:12:...|silver_produ

In [15]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.product_price and load_controller


In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------+--------------------+--------------------+--------------------+
|product_id| store|channel|list_price|cost_price|currency|effective_start_date|is_current|is_deleted|    source_file|          created_on|          created_by|         modified_on|         modified_by|
+----------+------+-------+----------+----------+--------+--------------------+----------+----------+---------------+--------------------+--------------------+--------------------+--------------------+
|     P1001|AMZ_US|    MKT|    500.00|    350.00|     USD|          2025-01-01|      true|     false|dim_product.csv|2026-01-29 01:12:...|silver_product_pr...|2026-01-29 01:12:...|silver_product_pr...|
|     P1002|  EBAY|    MKT|   1200.00|    900.00|     USD|          2025-02-15|      true|     false|dim_product.csv|2026-01-29 01:12:...|silver_product_pr...|2026-01-29 01:12:...|silver_produ

In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+-------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|   table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+-------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|product_price|full-load| overwrite|2026-01-29 01:12:...|    success|           3|2026-01-29 01:12:...|silver_product_pr...|2026-01-29 01:12:...|silver_product_pr...|
+------+-----------+-------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")##"bronze.dim_date_bronze")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |3            |
+-------+-------------+



In [19]:
spark.stop()